# Lab 2

In [ ]:
%pip install -q modal onnx onnxruntime pillow


In [ ]:
from __future__ import annotations

from collections import defaultdict
from contextlib import contextmanager, nullcontext
from pathlib import Path
from typing import Any
import copy
import json
import math
import os
import random
import time
import warnings

warnings.filterwarnings("ignore", message=r".*legacy TorchScript-based ONNX.*")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

try:
    import modal
except ImportError:
    modal = None

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

BICUBIC = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
MODAL_VOLUME_NAME = "lab2-phase7-data"
MODAL_MOUNT_ROOT = Path("/mnt/lab2-phase7-data")
RUN_ID = os.environ.get("LAB2_RUN_ID", time.strftime("%Y%m%d_%H%M%S"))
RUN_NAME = f"leaky_relu_last_run_{RUN_ID}"

MODEL_ID = "mixed_kernel_residual"
SEED = 255
BATCH_SIZE = 24
NUM_EPOCHS = 12
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
TRAIN_PATCH_SIZE = 224
EVAL_SIZE = 256
TRAIN_NUM_WORKERS = 0
EVAL_NUM_WORKERS = 0
USE_AMP = True
CHANNELS_LAST = True
RUN_TRAINING = True
RUN_ONNX_EXPORT = True
VERIFY_ONNX_EXPORT = True
LEAKY_RELU_SLOPE = 0.10

def looks_like_data_root(path: Path) -> bool:
    return (
        (path / "HR_train").exists() and (path / "LR_train").exists()
    ) or (
        (path / "train" / "HR").exists() and (path / "train" / "LR").exists()
    )


if not MODAL_MOUNT_ROOT.exists():
    raise RuntimeError(
        "Expected lab2-phase7-data mounted at /mnt/lab2-phase7-data in the Modal notebook."
    )

data_override = os.environ.get("LAB2_DATA_ROOT", "").strip()
if data_override:
    DATA_ROOT = Path(data_override).expanduser().resolve()
else:
    candidates = [MODAL_MOUNT_ROOT / "Data", MODAL_MOUNT_ROOT]
    DATA_ROOT = next((path for path in candidates if looks_like_data_root(path)), candidates[0])

if not looks_like_data_root(DATA_ROOT):
    raise FileNotFoundError(f"No paired Lab 2 data found under {DATA_ROOT}")

output_override = os.environ.get("LAB2_OUTPUT_DIR", "").strip()
OUTPUT_DIR = Path(output_override).expanduser().resolve() if output_override else (MODAL_MOUNT_ROOT / "runs" / RUN_NAME)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
EXPORT_DIR = OUTPUT_DIR / "exports"

for path in [OUTPUT_DIR, CHECKPOINT_DIR, EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(
    json.dumps(
        {
            "modal_volume": MODAL_VOLUME_NAME,
            "mount_root": str(MODAL_MOUNT_ROOT),
            "data_root": str(DATA_ROOT.resolve()),
            "output_dir": str(OUTPUT_DIR.resolve()),
            "model_id": MODEL_ID,
            "epochs": NUM_EPOCHS,
            "activation": f"LeakyReLU(slope={LEAKY_RELU_SLOPE:.2f})",
        },
        indent=2,
    )
)


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@contextmanager
def autocast_context(device: torch.device):
    if USE_AMP and device.type == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            yield
    else:
        with nullcontext():
            yield


def resolve_runs_volume():
    if modal is None:
        return None
    try:
        return modal.Volume.from_name(MODAL_VOLUME_NAME, create_if_missing=False)
    except Exception as exc:
        print(f"Modal volume handle unavailable: {exc}")
        return None


RUNS_VOLUME = resolve_runs_volume()


def commit_modal_volume(reason: str) -> None:
    if RUNS_VOLUME is None:
        return
    try:
        RUNS_VOLUME.commit()
        print(f"Committed volume: {reason}")
    except Exception as exc:
        print(f"Volume commit skipped: {exc}")


def pil_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def pil_to_tensor(img: Image.Image) -> torch.Tensor:
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return torch.from_numpy(arr)


def tensor_psnr(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    mse = torch.mean((a.float().clamp(0, 1) - b.float().clamp(0, 1)) ** 2, dim=(-3, -2, -1))
    mse = mse.clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


def collect_paired_by_subfolder(lr_root: Path, hr_root: Path) -> list[tuple[Path, Path, str]]:
    pairs: list[tuple[Path, Path, str]] = []
    if not lr_root.exists() or not hr_root.exists():
        return pairs
    for hr_dir in sorted(p for p in hr_root.iterdir() if p.is_dir()):
        suffix = hr_dir.name.replace("HR_train", "")
        lr_dir = lr_root / f"LR_train{suffix}"
        if not lr_dir.exists():
            continue
        hr_imgs = {p.stem: p for p in sorted(hr_dir.iterdir()) if p.suffix.lower() in IMAGE_SUFFIXES}
        lr_imgs = {p.stem: p for p in sorted(lr_dir.iterdir()) if p.suffix.lower() in IMAGE_SUFFIXES}
        for stem in sorted(set(hr_imgs) & set(lr_imgs)):
            pairs.append((lr_imgs[stem], hr_imgs[stem], f"{hr_dir.name}/{stem}"))
    return pairs


def collect_paired_flat(lr_dir: Path, hr_dir: Path) -> list[tuple[Path, Path, str]]:
    if not lr_dir.exists() or not hr_dir.exists():
        return []
    hr_imgs = {p.stem: p for p in sorted(hr_dir.iterdir()) if p.suffix.lower() in IMAGE_SUFFIXES}
    lr_imgs = {p.stem: p for p in sorted(lr_dir.iterdir()) if p.suffix.lower() in IMAGE_SUFFIXES}
    return [(lr_imgs[stem], hr_imgs[stem], stem) for stem in sorted(set(hr_imgs) & set(lr_imgs))]


def collect_train_pairs(data_root: Path) -> list[tuple[Path, Path, str]]:
    structured = collect_paired_by_subfolder(data_root / "LR_train", data_root / "HR_train")
    if structured:
        return structured
    return collect_paired_flat(data_root / "train" / "LR", data_root / "train" / "HR")


def collect_val_pairs(data_root: Path) -> list[tuple[Path, Path, str]]:
    for lr_dir, hr_dir in [
        (data_root / "LR_val", data_root / "HR_val"),
        (data_root / "val" / "LR_val", data_root / "val" / "HR_val"),
        (data_root / "val" / "LR", data_root / "val" / "HR"),
    ]:
        pairs = collect_paired_flat(lr_dir, hr_dir)
        if pairs:
            return pairs
    return []


def random_crop_pair(lr_img: Image.Image, hr_img: Image.Image, size: int, rng: random.Random) -> tuple[Image.Image, Image.Image]:
    width, height = lr_img.size
    if min(width, height) < size:
        lr_img = ImageOps.fit(lr_img, (size, size), method=BICUBIC)
        hr_img = ImageOps.fit(hr_img, (size, size), method=BICUBIC)
        return lr_img, hr_img
    left = rng.randint(0, width - size)
    top = rng.randint(0, height - size)
    box = (left, top, left + size, top + size)
    return lr_img.crop(box), hr_img.crop(box)


def augment_pair(lr_img: Image.Image, hr_img: Image.Image, rng: random.Random) -> tuple[Image.Image, Image.Image]:
    if rng.random() < 0.5:
        lr_img = ImageOps.mirror(lr_img)
        hr_img = ImageOps.mirror(hr_img)
    if rng.random() < 0.5:
        lr_img = ImageOps.flip(lr_img)
        hr_img = ImageOps.flip(hr_img)
    rotations = rng.randint(0, 3)
    if rotations:
        angle = 90 * rotations
        lr_img = lr_img.rotate(angle)
        hr_img = hr_img.rotate(angle)
    return lr_img, hr_img


class PairedSRDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path, str]], train: bool, seed: int):
        self.pairs = pairs
        self.train = train
        self.seed = seed

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        lr_path, hr_path, name = self.pairs[idx]
        lr_img = pil_rgb(lr_path)
        hr_img = pil_rgb(hr_path)
        if self.train:
            rng = random.Random(self.seed + idx)
            lr_img, hr_img = random_crop_pair(lr_img, hr_img, TRAIN_PATCH_SIZE, rng)
            lr_img, hr_img = augment_pair(lr_img, hr_img, rng)
        else:
            lr_img = ImageOps.fit(lr_img, (EVAL_SIZE, EVAL_SIZE), method=BICUBIC)
            hr_img = ImageOps.fit(hr_img, (EVAL_SIZE, EVAL_SIZE), method=BICUBIC)
        return pil_to_tensor(lr_img), pil_to_tensor(hr_img), name


def loader_kwargs(num_workers: int, pin_memory: bool) -> dict[str, Any]:
    kwargs: dict[str, Any] = {"num_workers": num_workers, "pin_memory": pin_memory}
    if num_workers > 0:
        kwargs["persistent_workers"] = True
    return kwargs


def make_dataloaders(
    train_pairs: list[tuple[Path, Path, str]],
    val_pairs: list[tuple[Path, Path, str]],
    device: torch.device,
) -> tuple[DataLoader, DataLoader]:
    pin_memory = device.type == "cuda"
    train_loader = DataLoader(
        PairedSRDataset(train_pairs, train=True, seed=SEED),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        **loader_kwargs(TRAIN_NUM_WORKERS, pin_memory),
    )
    val_loader = DataLoader(
        PairedSRDataset(val_pairs, train=False, seed=SEED),
        batch_size=BATCH_SIZE,
        shuffle=False,
        **loader_kwargs(EVAL_NUM_WORKERS, pin_memory),
    )
    return train_loader, val_loader

In [12]:
def make_activation() -> nn.Module:
    return nn.LeakyReLU(negative_slope=LEAKY_RELU_SLOPE, inplace=True)


class ConvLeakyReLUBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3, groups: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(
            channels,
            channels,
            kernel_size,
            padding=kernel_size // 2,
            groups=groups,
            bias=True,
        )
        self.act = make_activation()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.conv(x))


def init_tail_small(tail: nn.Conv2d, scale: float = 1e-3) -> None:
    nn.init.kaiming_normal_(tail.weight, nonlinearity="linear")
    tail.weight.data.mul_(scale)
    if tail.bias is not None:
        nn.init.zeros_(tail.bias)


class ResidualConvSR(nn.Module):
    def __init__(self, channels: int = 96, kernel_pattern: tuple[int, ...] = (3,) * 18):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=True),
            make_activation(),
        )
        self.body = nn.Sequential(*[ConvLeakyReLUBlock(channels, kernel_size=k) for k in kernel_pattern])
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        init_tail_small(self.tail)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        return self.tail(self.body(self.stem(x)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


class ExpandedDepthwiseBlock(nn.Module):
    def __init__(self, channels: int, expansion: int, kernel_size: int):
        super().__init__()
        hidden = channels * expansion
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=True),
            make_activation(),
            nn.Conv2d(
                hidden,
                hidden,
                kernel_size,
                padding=kernel_size // 2,
                groups=hidden,
                bias=True,
            ),
            make_activation(),
            nn.Conv2d(hidden, channels, 1, bias=True),
            make_activation(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ExpandedDepthwiseResidualSR(nn.Module):
    def __init__(self, channels: int = 128, expansion: int = 2, num_blocks: int = 12, kernel_size: int = 9):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=True),
            make_activation(),
        )
        self.body = nn.Sequential(*[ExpandedDepthwiseBlock(channels, expansion, kernel_size) for _ in range(num_blocks)])
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        init_tail_small(self.tail)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        return self.tail(self.body(self.stem(x)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


class HybridLargeFineResidualSR(nn.Module):
    def __init__(
        self,
        channels: int = 128,
        expansion: int = 2,
        dw_blocks: int = 8,
        dw_kernel_size: int = 9,
        fine_kernel_pattern: tuple[int, ...] = (3, 5, 3, 5, 3, 5, 3, 5, 3, 5),
    ):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=True),
            make_activation(),
        )
        body_layers: list[nn.Module] = []
        body_layers.extend(
            ExpandedDepthwiseBlock(channels, expansion, dw_kernel_size)
            for _ in range(dw_blocks)
        )
        body_layers.extend(
            ConvLeakyReLUBlock(channels, kernel_size=k) for k in fine_kernel_pattern
        )
        self.body = nn.Sequential(*body_layers)
        self.tail = nn.Conv2d(channels, 3, 3, padding=1, bias=True)
        init_tail_small(self.tail)

    def predict_delta(self, x: torch.Tensor) -> torch.Tensor:
        return self.tail(self.body(self.stem(x)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.predict_delta(x)


MODEL_REGISTRY = {
    "nonorm_residual": {
        "display_name": "ResidualConvSR_3x3x18",
        "build_model": lambda **cfg: ResidualConvSR(**cfg),
        "model_cfg": {"channels": 96, "kernel_pattern": tuple([3] * 18)},
    },
    "mixed_kernel_residual": {
        "display_name": "ResidualConvSR_mixed_3_3_5",
        "build_model": lambda **cfg: ResidualConvSR(**cfg),
        "model_cfg": {"channels": 96, "kernel_pattern": tuple([3, 3, 5] * 6)},
    },
    "expanded_dw_large_residual": {
        "display_name": "ExpandedDepthwiseResidualSR_9x9x12",
        "build_model": lambda **cfg: ExpandedDepthwiseResidualSR(**cfg),
        "model_cfg": {
            "channels": 128,
            "expansion": 2,
            "num_blocks": 12,
            "kernel_size": 9,
        },
    },
    "expanded_dw_large_deep": {
        "display_name": "ExpandedDepthwiseResidualSR_7x7x18",
        "build_model": lambda **cfg: ExpandedDepthwiseResidualSR(**cfg),
        "model_cfg": {
            "channels": 128,
            "expansion": 2,
            "num_blocks": 18,
            "kernel_size": 7,
        },
    },
    "mixed_kernel_residual_wide": {
        "display_name": "ResidualConvSR_wide_mixed_3_3_5",
        "build_model": lambda **cfg: ResidualConvSR(**cfg),
        "model_cfg": {"channels": 128, "kernel_pattern": tuple([3, 3, 5] * 8)},
    },
    "expanded_dw_large_v2": {
        "display_name": "ExpandedDepthwiseResidualSR_9x9x14_ch160",
        "build_model": lambda **cfg: ExpandedDepthwiseResidualSR(**cfg),
        "model_cfg": {
            "channels": 160,
            "expansion": 2,
            "num_blocks": 14,
            "kernel_size": 9,
        },
    },
    "hybrid_large_fine_residual": {
        "display_name": "HybridLargeFineResidualSR",
        "build_model": lambda **cfg: HybridLargeFineResidualSR(**cfg),
        "model_cfg": {
            "channels": 128,
            "expansion": 2,
            "dw_blocks": 8,
            "dw_kernel_size": 9,
            "fine_kernel_pattern": (3, 5, 3, 5, 3, 5, 3, 5, 3, 5),
        },
    },
}


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def assert_module_compatible(model: nn.Module) -> None:
    allowed_leaf_types = (nn.Conv2d, nn.LeakyReLU)
    forbidden_types = (
        nn.ReLU,
        nn.Sigmoid,
        nn.Softmax,
        nn.LayerNorm,
        nn.GroupNorm,
        nn.BatchNorm2d,
        nn.Dropout,
        nn.Dropout2d,
        nn.AdaptiveAvgPool2d,
        nn.Hardsigmoid,
    )
    for name, module in model.named_modules():
        if isinstance(module, forbidden_types):
            raise TypeError(f"Forbidden module at {name}: {module.__class__.__name__}")
        if name and len(list(module.children())) == 0 and not isinstance(module, allowed_leaf_types):
            raise TypeError(f"Unsupported leaf module at {name}: {module.__class__.__name__}")


def summarize_model_modules(model: nn.Module) -> dict[str, int]:
    counts: dict[str, int] = defaultdict(int)
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            if module.groups == module.in_channels == module.out_channels:
                counts["DepthwiseConv2d"] += 1
            elif module.groups > 1:
                counts["GroupedConv2d"] += 1
            else:
                counts["Conv2d"] += 1
        elif isinstance(module, nn.LeakyReLU):
            counts["LeakyReLU"] += 1
    return dict(counts)


def get_model_spec(model_id: str) -> dict[str, Any]:
    if model_id not in MODEL_REGISTRY:
        raise KeyError(f"Unknown MODEL_ID={model_id}; choose {sorted(MODEL_REGISTRY)}")
    spec = copy.deepcopy(MODEL_REGISTRY[model_id])
    probe = spec["build_model"](**spec["model_cfg"])
    assert_module_compatible(probe)
    spec["params"] = count_parameters(probe)
    spec["module_ops"] = summarize_model_modules(probe)
    return spec

In [13]:
def move_batch(
    lr_img: torch.Tensor,
    hr_img: torch.Tensor,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    lr_img = lr_img.to(device, non_blocking=True)
    hr_img = hr_img.to(device, non_blocking=True)
    if CHANNELS_LAST and device.type == "cuda":
        lr_img = lr_img.contiguous(memory_format=torch.channels_last)
        hr_img = hr_img.contiguous(memory_format=torch.channels_last)
    return lr_img, hr_img


def residual_target_l1_loss(sr_pred: torch.Tensor, lr_img: torch.Tensor, hr_img: torch.Tensor) -> torch.Tensor:
    return F.l1_loss(sr_pred - lr_img, hr_img - lr_img)


def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return tensor_psnr(pred.clamp(0.0, 1.0), target.clamp(0.0, 1.0))


def verify_residual_l1_batch(model: nn.Module, loader: DataLoader, device: torch.device) -> None:
    lr_img, hr_img, _ = next(iter(loader))
    lr_img, hr_img = move_batch(lr_img, hr_img, device)
    pred = model(lr_img)
    loss = residual_target_l1_loss(pred, lr_img, hr_img)
    manual = F.l1_loss(pred - lr_img, hr_img - lr_img)
    torch.testing.assert_close(loss, manual)
    print("Residual L1 check passed.")


def save_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    metrics: dict[str, Any],
    best_metric: float,
    model_cfg: dict[str, Any],
) -> None:
    state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "best_metric": best_metric,
        "model_cfg": model_cfg,
        "model_id": MODEL_ID,
        "saved_at": time.time(),
    }
    torch.save(state, path)


def load_weights(
    model: nn.Module,
    checkpoint_path: Path,
    map_location: str | torch.device = "cpu",
) -> dict[str, Any]:
    ckpt = torch.load(checkpoint_path, map_location=map_location)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict)
    return ckpt


def make_grad_scaler(device: torch.device):
    if not (USE_AMP and device.type == "cuda"):
        return None
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=True)
    return torch.cuda.amp.GradScaler(enabled=True)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: Any,
    device: torch.device,
) -> dict[str, float]:
    model.train()
    total_loss = 0.0
    total_psnr = 0.0
    count = 0
    for lr_img, hr_img, _ in loader:
        lr_img, hr_img = move_batch(lr_img, hr_img, device)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with autocast_context(device):
                pred = model(lr_img)
                loss = residual_target_l1_loss(pred, lr_img, hr_img)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with autocast_context(device):
                pred = model(lr_img)
                loss = residual_target_l1_loss(pred, lr_img, hr_img)
            loss.backward()
            optimizer.step()
        batch_size = lr_img.size(0)
        total_loss += float(loss.item()) * batch_size
        with torch.no_grad():
            total_psnr += compute_psnr(pred.detach(), hr_img).sum().item()
        count += batch_size
    return {
        "train_loss": total_loss / max(1, count),
        "train_psnr": total_psnr / max(1, count),
    }


@torch.no_grad()
def evaluate_loader(model: nn.Module, loader: DataLoader, device: torch.device) -> dict[str, Any]:
    model.eval()
    total_loss = 0.0
    total_psnr = 0.0
    total_input_psnr = 0.0
    total_delta = 0.0
    count = 0
    worst_examples: list[dict[str, Any]] = []
    for lr_img, hr_img, names in loader:
        lr_img, hr_img = move_batch(lr_img, hr_img, device)
        with autocast_context(device):
            pred = model(lr_img)
            loss = residual_target_l1_loss(pred, lr_img, hr_img)
        pred_psnr = compute_psnr(pred, hr_img)
        input_psnr = compute_psnr(lr_img, hr_img)
        delta = pred_psnr - input_psnr
        batch_size = lr_img.size(0)
        total_loss += float(loss.item()) * batch_size
        total_psnr += pred_psnr.sum().item()
        total_input_psnr += input_psnr.sum().item()
        total_delta += delta.sum().item()
        count += batch_size
        for i in range(batch_size):
            worst_examples.append(
                {
                    "name": str(names[i]),
                    "input_psnr": float(input_psnr[i].item()),
                    "val_psnr": float(pred_psnr[i].item()),
                    "delta_psnr": float(delta[i].item()),
                }
            )
    if count == 0:
        raise RuntimeError("Validation loader produced no samples")
    worst_examples = sorted(worst_examples, key=lambda row: row["val_psnr"])[:10]
    return {
        "val_loss": total_loss / count,
        "val_psnr": total_psnr / count,
        "val_input_psnr": total_input_psnr / count,
        "val_delta_psnr": total_delta / count,
        "val_worst10": worst_examples,
    }


def train_model(
    spec: dict[str, Any],
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
) -> Path:
    model = spec["build_model"](**spec["model_cfg"]).to(device)
    if CHANNELS_LAST and device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, NUM_EPOCHS))
    scaler = make_grad_scaler(device)

    verify_residual_l1_batch(model, train_loader, device)

    best_metric = float("-inf")
    best_path = CHECKPOINT_DIR / "best.pt"
    last_path = CHECKPOINT_DIR / "last.pt"
    print(f"{'epoch':>5} | {'lr':>9} | {'train_loss':>10} | {'train_psnr':>10} | {'val_psnr':>9} | {'delta':>8}")
    print("-" * 66)
    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, device)
        val_metrics = evaluate_loader(model, val_loader, device)
        metrics = {
            "epoch": epoch,
            "lr": float(optimizer.param_groups[0]["lr"]),
            **train_metrics,
            **val_metrics,
        }
        save_checkpoint(last_path, model, optimizer, epoch, metrics, best_metric, spec["model_cfg"])
        if val_metrics["val_psnr"] > best_metric:
            best_metric = float(val_metrics["val_psnr"])
            save_checkpoint(best_path, model, optimizer, epoch, metrics, best_metric, spec["model_cfg"])
        print(
            f"{epoch:5d} | {metrics['lr']:9.2e} | {metrics['train_loss']:10.5f} | {metrics['train_psnr']:10.3f} | {metrics['val_psnr']:9.3f} | {metrics['val_delta_psnr']:8.3f}"
        )
        scheduler.step()
    print(f"Best checkpoint: {best_path}")
    return best_path


def export_to_onnx(
    spec: dict[str, Any],
    checkpoint_path: Path,
    onnx_path: Path,
    device: torch.device,
    sample_loader: DataLoader | None = None,
) -> Path:
    model = spec["build_model"](**spec["model_cfg"]).to(device)
    load_weights(model, checkpoint_path, map_location=device)
    model.eval()
    dummy = torch.randn(1, 3, EVAL_SIZE, EVAL_SIZE, device=device)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    export_kwargs = {
        "export_params": True,
        "opset_version": 13,
        "do_constant_folding": True,
        "input_names": ["input"],
        "output_names": ["output"],
    }
    try:
        torch.onnx.export(model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(model, dummy, str(onnx_path), **export_kwargs)
    if onnx is not None:
        onnx.checker.check_model(onnx.load(str(onnx_path)))
        print("ONNX checker passed.")
    if VERIFY_ONNX_EXPORT and sample_loader is not None and ort is not None:
        sample_lr, _, _ = next(iter(sample_loader))
        sample_lr = sample_lr[:1].to(device)
        with torch.no_grad():
            torch_out = model(sample_lr).detach().cpu().numpy()
        session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_out = session.run(["output"], {"input": sample_lr.cpu().numpy()})[0]
        diff = np.abs(torch_out - ort_out)
        print(f"ONNX Runtime parity: max_diff={diff.max():.8f}, mean_diff={diff.mean():.8f}")
    print(f"Exported ONNX: {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB)")
    return onnx_path

In [ ]:
set_seed(SEED)
device = resolve_device()
train_pairs = collect_train_pairs(DATA_ROOT)
val_pairs = collect_val_pairs(DATA_ROOT)
if not train_pairs:
    raise FileNotFoundError(f"No paired training images found under {DATA_ROOT.resolve()}")
if not val_pairs:
    raise FileNotFoundError(f"No paired validation images found under {DATA_ROOT.resolve()}")

print(f"Using device: {device}")
print(f"Paired train pairs: {len(train_pairs)}")
print(f"Paired val pairs: {len(val_pairs)}")

train_loader, val_loader = make_dataloaders(train_pairs, val_pairs, device)
spec = get_model_spec(MODEL_ID)
print(f"Activation: LeakyReLU(slope={LEAKY_RELU_SLOPE:.2f})")
print(f"Parameters: {spec['params']:,}")
print(f"Module summary: {spec['module_ops']}")

best_checkpoint: Path | None = None
if RUN_TRAINING:
    best_checkpoint = train_model(spec, train_loader, val_loader, device)
    commit_modal_volume("post_training")
else:
    candidate = CHECKPOINT_DIR / "best.pt"
    if candidate.exists():
        best_checkpoint = candidate
        print(f"Using existing checkpoint: {best_checkpoint}")

if RUN_ONNX_EXPORT:
    if best_checkpoint is None or not best_checkpoint.exists():
        raise FileNotFoundError("RUN_ONNX_EXPORT=True but no best checkpoint is available.")
    export_to_onnx(spec, best_checkpoint, EXPORT_DIR / "best.onnx", device, sample_loader=val_loader)
    commit_modal_volume("post_export")
